In [9]:
def is_valid(var, value, assignment):

    if value in assignment.values():
        return False

    temp = assignment.copy()

    temp[var] = value

    if "W" in temp and "D" in temp and temp["W"] >= temp["D"]:
        return False

    if "H" in temp and temp["H"] == 3:
        return False

    if "A" in temp and "H" in temp and temp["H"] >= temp["A"]:
        return False

    if "O" in temp and temp["O"] != 5:
        return False

    if "W" in temp and "D" in temp and abs(temp["W"] - temp["D"]) == 1:
        return False

    return True


def csp(variables, domain, assignment):
    if len(assignment) == len(variables):
        return assignment

    for var in variables:
        if var not in assignment:
            cur_var = var
            break

    for value in domain:
        if is_valid(cur_var, value, assignment):
            assignment[cur_var] = value

            res = csp(variables, domain, assignment)

            if res is not None:
                return res

            del assignment[cur_var]

    return None


variables = ["W", "D", "H", "A", "O"]

domains = range(1, 6)

solutions = csp(variables, domains, {})

print(solutions)

{'W': 1, 'D': 3, 'H': 2, 'A': 4, 'O': 5}


In [1]:
import itertools

# The domain (time slots) available for the devices
time_slots = [1, 2, 3, 4, 5]

solutions = []

# Constraint a: All devices run at different times.
# We handle this by generating permutations of the 5 time slots.
# The variables mapped are W, D, H, A, O.
for W, D, H, A, O in itertools.permutations(time_slots):

    # Constraint b: Washing machine runs before the dryer (Dishwasher).
    if not (W < D):
        continue

    # Constraint c: Heater cannot run at time 3.
    if H == 3:
        continue

    # Constraint d: Air conditioner runs after the heater.
    if not (A > H):
        continue

    # Constraint e: Oven must run in time slot 5.
    if O != 5:
        continue

    # Constraint f: Dishwasher cannot be adjacent to Washing Machine.
    if abs(D - W) == 1:
        continue

    # If all constraints are met, a valid solution is appended!
    solutions.append({"W": W, "D": D, "H": H, "A": A, "O": O})

for i, sol in enumerate(solutions, 1):
    print(f"Solution {i}: {sol}")

Solution 1: {'W': 1, 'D': 3, 'H': 2, 'A': 4, 'O': 5}
Solution 2: {'W': 1, 'D': 4, 'H': 2, 'A': 3, 'O': 5}
Solution 3: {'W': 2, 'D': 4, 'H': 1, 'A': 3, 'O': 5}


In [2]:
import itertools

signals = range(1, 6)

sol = []

for i1, i2, i3, i4, i5 in itertools.permutations(signals):
    if i1 > i3:
        continue

    if i5 == 1:
        continue

    if abs(i2 - i4) > 2:
        continue

    if i3 != 3:
        continue

    sol.append({"i1": i1, "i2": i2, "i3": i3, "i4": i4, "i5": i5})

for i, s in enumerate(sol, 1):
    print(f"Solution {i}: {s}")

Solution 1: {'i1': 1, 'i2': 2, 'i3': 3, 'i4': 4, 'i5': 5}
Solution 2: {'i1': 1, 'i2': 4, 'i3': 3, 'i4': 2, 'i5': 5}
Solution 3: {'i1': 1, 'i2': 4, 'i3': 3, 'i4': 5, 'i5': 2}
Solution 4: {'i1': 1, 'i2': 5, 'i3': 3, 'i4': 4, 'i5': 2}


In [3]:
class CSP:
    def __init__(self, variables, domains):
        self.variables = variables
        self.domains = domains
        self.constraints = []

    def add_constraint(self, constraint_func):
        self.constraints.append(constraint_func)

    # Checks if the current assignments break any rules
    def is_consistent(self, assignment):
        for constraint in self.constraints:
            if not constraint(assignment):
                return False
        return True

    # Core recursive CSP Backtracking Algorithm
    def backtrack(self, assignment=None):
        if assignment is None:
            assignment = {}

        # Base case: if all variables are assigned, we found a valid solution
        if len(assignment) == len(self.variables):
            return [assignment.copy()]

        # Pick the next unassigned variable
        unassigned = [v for v in self.variables if v not in assignment]
        var = unassigned[0]

        solutions = []
        for value in self.domains[var]:
            assignment[var] = value
            # Only proceed deeper if the current path is valid
            if self.is_consistent(assignment):
                result = self.backtrack(assignment)
                if result:
                    solutions.extend(result)
            # Backtrack: remove the assignment and try the next value
            del assignment[var]

        return solutions


# --- Setup Variables and Domains ---
variables = ["I1", "I2", "I3", "I4", "I5"]
domains = {var: [1, 2, 3, 4, 5] for var in variables}

csp = CSP(variables, domains)


# --- Add Constraints ---
# 1. All intersections assigned different signal phases (1-5)
def all_different(assignment):
    values = list(assignment.values())
    return len(values) == len(set(values))


csp.add_constraint(all_different)


# 2. I1 must be before I3
def i1_before_i3(assignment):
    if "I1" in assignment and "I3" in assignment:
        return assignment["I1"] < assignment["I3"]
    return True


csp.add_constraint(i1_before_i3)


# 3. I5 cannot be phase 1
def i5_not_1(assignment):
    if "I5" in assignment:
        return assignment["I5"] != 1
    return True


csp.add_constraint(i5_not_1)


# 4. I2 and I4 cannot differ by more than 2
def i2_i4_diff(assignment):
    if "I2" in assignment and "I4" in assignment:
        return abs(assignment["I2"] - assignment["I4"]) <= 2
    return True


csp.add_constraint(i2_i4_diff)


# 5. I3 must be phase 3
def i3_is_3(assignment):
    if "I3" in assignment:
        return assignment["I3"] == 3
    return True


csp.add_constraint(i3_is_3)

# --- Run the Algorithm ---
solutions = csp.backtrack()

print(f"Total valid combinations found: {len(solutions)}")
for i, sol in enumerate(solutions, 1):
    print(f"Solution {i}: {sol}")

Total valid combinations found: 4
Solution 1: {'I1': 1, 'I2': 2, 'I3': 3, 'I4': 4, 'I5': 5}
Solution 2: {'I1': 1, 'I2': 4, 'I3': 3, 'I4': 2, 'I5': 5}
Solution 3: {'I1': 1, 'I2': 4, 'I3': 3, 'I4': 5, 'I5': 2}
Solution 4: {'I1': 1, 'I2': 5, 'I3': 3, 'I4': 4, 'I5': 2}


In [5]:
import itertools

slots = range(1, 6)

sols = []

for a, b, c, d, e in itertools.permutations(slots):
    if a > b:
        continue

    if c == 2:
        continue

    if d != c + 1:
        continue

    if e != 5:
        continue

    if abs(b - d) == 1:
        continue

    sols.append({"a": a, "b": b, "c": c, "d": d, "e": e})

for i, so in enumerate(sols, 1):
    print(f"solution {i}: {so}")

solution 1: {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
solution 2: {'a': 3, 'b': 4, 'c': 1, 'd': 2, 'e': 5}


In [10]:
from ortools.sat.python import cp_model


model = cp_model.CpModel()

w = model.new_int_var(1, 5, "w")
d = model.new_int_var(1, 5, "d")
h = model.new_int_var(1, 5, "h")
a = model.new_int_var(1, 5, "a")
o = model.new_int_var(1, 5, "o")

model.add_all_different([w, d, h, a, o])
model.add(w < d)
model.add(h != 3)
model.add(a > h)
model.add(o == 5)

diff_dw = model.new_int_var(0, 5, "diff_dw")
model.add_abs_equality(diff_dw, d - w)
model.add(diff_dw != 1)


solver = cp_model.CpSolver()
if solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"solution: w:{solver.value(w)}, d:{solver.value(d)}, h:{solver.value(h)}, a:{solver.value(a)}, o:{solver.value(o)},"
    )

solution: w:1, d:4, h:2, a:3, o:5,


In [14]:
from ortools.sat.python import cp_model

bot = cp_model.CpModel()

i1 = bot.new_int_var(1, 5, "i1")
i2 = bot.new_int_var(1, 5, "i2")
i3 = bot.new_int_var(1, 5, "i3")
i4 = bot.new_int_var(1, 5, "i4")
i5 = bot.new_int_var(1, 5, "i5")


bot.AddAllDifferent([i1, i2, i3, i4, i5])
bot.Add(i1 < i3)
bot.Add(i5 != 1)
bot.Add(i3 == 3)

diff_24 = bot.NewIntVar(0, 5, "diff_24")
bot.AddAbsEquality(diff_24, i2 - i4)
bot.Add(diff_24 <= 2)


xol = cp_model.CpSolver()
if xol.solve(bot) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"i1:{xol.Value(i1)}, i2:{xol.Value(i2)}, i3:{xol.Value(i3)}, i4:{xol.Value(i4)}, i5:{xol.Value(i5)}"
    )

i1:1, i2:4, i3:3, i4:5, i5:2


In [15]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

a = model.new_int_var(1, 5, "a")
b = model.new_int_var(1, 5, "b")
c = model.new_int_var(1, 5, "c")
d = model.new_int_var(1, 5, "d")
e = model.new_int_var(1, 5, "e")

model.AddAllDifferent([a, b, c, d, e])
model.Add(a < b)
model.Add(c != 2)
model.Add(d == c + 1)
model.Add(e == 5)

diff_bd = model.new_int_var(0, 5, "diff_bd")
model.AddAbsEquality(diff_bd, b - d)
model.add(diff_bd != 1)

solver = cp_model.CpSolver()
if solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"a:{solver.value(a)}, b:{solver.value(b)}, c:{solver.value(c)}, d:{solver.value(d)}, e:{solver.value(e)}, "
    )

a:3, b:4, c:1, d:2, e:5, 


In [16]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

p1 = model.new_int_var(1, 5, "p1")
p2 = model.new_int_var(1, 5, "p2")
p3 = model.new_int_var(1, 5, "p3")
p4 = model.new_int_var(1, 5, "p4")
p5 = model.new_int_var(1, 5, "p5")

model.add_all_different([a, b, c, d, e])
model.add(p1 < p3)
model.add(p2 != 1)

diff_45 = model.new_int_var(0, 5, "diff_45")
model.add_abs_equality(diff_45, p4 - p5)
model.add(diff_45 == 1)

model.add(p3 == 4)

diff_12 = model.new_int_var(0, 5, "diff_12")
model.add_abs_equality(diff_12, p1 - p2)
model.add(diff_12 != 1)

solver = cp_model.CpSolver()

if solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"a:{solver.value(a)}, b:{solver.value(b)}, c:{solver.value(c)}, d:{solver.value(d)}, e:{solver.value(e)}, "
    )

a:3, b:5, c:4, d:1, e:2, 


In [1]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

d1 = model.new_int_var(1, 5, "d1")
d2 = model.new_int_var(1, 5, "d2")
d3 = model.new_int_var(1, 5, "d3")
d4 = model.new_int_var(1, 5, "d4")
d5 = model.new_int_var(1, 5, "d5")


model.add_all_different([d1, d2, d3, d4, d5])
model.add(d1 < d3)
model.add(d2 != 4)
model.add(d5 == d4 + 1)
model.add_allowed_assignments([d3], [[2], [3]])

diff_15 = model.new_int_var(0, 5, "diff_15")
model.add_abs_equality(diff_15, d1 - d5)
model.add(diff_15 != 1)

solver = cp_model.CpSolver()

if solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"d1:{solver.value(d1)}, d2:{solver.value(d2)}, d3:{solver.value(d3)}, d4:{solver.value(d4)}, d5:{solver.value(d5)},"
    )

d1:1, d2:5, d3:2, d4:3, d5:4,


In [2]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

c1 = model.new_int_var(1, 5, "c1")
c2 = model.new_int_var(1, 5, "c2")
c3 = model.new_int_var(1, 5, "c3")
c4 = model.new_int_var(1, 5, "c4")
c5 = model.new_int_var(1, 5, "c5")

model.add_all_different([c1, c2, c3, c4, c5])
model.add(c1 < c4)
model.Add(c2 != 3)

diff_35 = model.new_int_var(0, 5, "diff_35")
model.AddAbsEquality(diff_35, c3 - c5)
model.Add(diff_35 == 1)

model.Add(c4 == 5)

diff_12 = model.new_int_var(0, 5, "diff_12")
model.add_abs_equality(diff_12, c1 - c2)
model.Add(diff_12 != 1)


solver = cp_model.CpSolver()

if solver.solve(model) in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print(
        f"c1: {solver.value(c1)}, c2: {solver.value(c2)}, c3: {solver.value(c3)}, c4: {solver.value(c4)}, c5: {solver.value(c5)}, "
    )

c1: 4, c2: 1, c3: 3, c4: 5, c5: 2, 
